<a href="https://colab.research.google.com/github/lyntos/Python-MS/blob/main/03_Coding_Lab_7_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



Week 3 Coding Lab - 7.24

In [ ]:
"""7.24 (AI Project: Introducing Heuristic Programming with the Knight’s
Tour) An interesting puzzler for chess buffs is the Knight’s Tour problem,
originally proposed by the mathematician Euler. Can the knight piece
move around an empty chessboard and touch each of the 64 squares once
and only once? We study this intriguing problem in depth here.
The knight makes only L-shaped moves (two spaces in one direction and
one space in a perpendicular direction). Thus, as shown in the figure
below, from a square near the middle of an empty chessboard, the knight
(labeled K) can make eight different moves (numbered 0 through 7)."""

"""
Description:
This script attempts the Knight’s Tour on an 8-by-8 chessboard using:
1) A basic random-move approach (no heuristic), which records how many
   squares the knight visits before it gets stuck.
2) A heuristic approach using an "accessibility" matrix (Warnsdorff-style),
   where the knight always moves to the available square with the lowest
   accessibility number. As squares are visited, the script reduces the
   accessibility of squares that are reachable from the knight's new position.

Input:
- None (start position is chosen in code; you can change it).

Output:
- The board showing the move number in each visited square.
- The number of moves completed.
- For the heuristic version, optionally tests all 64 starting squares and
  counts how many full tours (64 moves) are achieved.
"""

import numpy as np
import random

# Knight move offsets (move numbers 0..7)
horizontal = np.array([2, 1, -1, -2, -2, -1, 1, 2])
vertical = np.array([-1, -2, -2, -1, 1, 2, 2, 1])

BOARD_SIZE = 8


def in_bounds(r, c):
  return 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE


def display_board(board):
  width = len(str(board.max()))
  for r in range(BOARD_SIZE):
    for c in range(BOARD_SIZE):
      print(f"{board[r, c]:>{width}}", end=" ")
    print()
  print()


def attempt_knights_tour_random(start_row, start_col):
  board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)

  current_row = start_row
  current_col = start_col
  move_number = 1
  board[current_row, current_col] = move_number

  while True:
    possible_moves = []

    for move in range(8):
      next_row = current_row + vertical[move]
      next_col = current_col + horizontal[move]

      if in_bounds(next_row, next_col) and board[next_row, next_col] == 0:
        possible_moves.append(move)

    if len(possible_moves) == 0:
      break

    move = random.choice(possible_moves)
    current_row += vertical[move]
    current_col += horizontal[move]

    move_number += 1
    board[current_row, current_col] = move_number

  return board, move_number


def initial_accessibility():
  # Standard accessibility matrix from the book
  return np.array([
    [2, 3, 4, 4, 4, 4, 3, 2],
    [3, 4, 6, 6, 6, 6, 4, 3],
    [4, 6, 8, 8, 8, 8, 6, 4],
    [4, 6, 8, 8, 8, 8, 6, 4],
    [4, 6, 8, 8, 8, 8, 6, 4],
    [4, 6, 8, 8, 8, 8, 6, 4],
    [3, 4, 6, 6, 6, 6, 4, 3],
    [2, 3, 4, 4, 4, 4, 3, 2]
  ], dtype=int)


def update_accessibility(access, row, col, board):
  # reduce accessibility of squares that are reachable from (row, col)
  for move in range(8):
    r = row + vertical[move]
    c = col + horizontal[move]
    if in_bounds(r, c) and board[r, c] == 0:
      access[r, c] -= 1


def attempt_knights_tour_heuristic(start_row, start_col):
  board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
  access = initial_accessibility()

  current_row = start_row
  current_col = start_col
  move_number = 1
  board[current_row, current_col] = move_number

  # visited squares should not be selected again
  access[current_row, current_col] = 99

  # reduce accessibility of squares reachable from the starting position
  update_accessibility(access, current_row, current_col, board)

  while move_number < 64:
    best_moves = []
    best_value = 10**9

    for move in range(8):
      next_row = current_row + vertical[move]
      next_col = current_col + horizontal[move]

      if in_bounds(next_row, next_col) and board[next_row, next_col] == 0:
        value = access[next_row, next_col]
        if value < best_value:
          best_value = value
          best_moves = [(next_row, next_col)]
        elif value == best_value:
          best_moves.append((next_row, next_col))

    if len(best_moves) == 0:
      break

    # tie-break: choose randomly among best moves
    next_row, next_col = random.choice(best_moves)

    current_row = next_row
    current_col = next_col
    move_number += 1
    board[current_row, current_col] = move_number

    access[current_row, current_col] = 99
    update_accessibility(access, current_row, current_col, board)

  return board, move_number


# -------------------------
# Run a single demo
# -------------------------
start_row = 0
start_col = 0

print("Random-move Knight's Tour attempt:")
random_board, random_moves = attempt_knights_tour_random(start_row, start_col)
display_board(random_board)
print("Moves completed:", random_moves)
print()

print("Heuristic Knight's Tour attempt:")
heur_board, heur_moves = attempt_knights_tour_heuristic(start_row, start_col)
display_board(heur_board)
print("Moves completed:", heur_moves)
print()

# -------------------------
# Test all 64 starting squares for full tours (heuristic)
# -------------------------
full_tours = 0

for r in range(BOARD_SIZE):
  for c in range(BOARD_SIZE):
    _, moves = attempt_knights_tour_heuristic(r, c)
    if moves == 64:
      full_tours += 1

print("Heuristic full tours (out of 64 starts):", full_tours)

Random-move Knight's Tour attempt:
 1  0  3  0  0  0  0  0 
 4  7  0  0  0  0 30  0 
15  2  9  6  0 28 25  0 
 8  5 14 27 24 31  0 29 
 0 16 23 10  0 26 43 32 
22 13 20 17 44 35 38 41 
 0 18 11  0 39 42 33 36 
12 21  0 19 34 37 40  0 

Moves completed: 44

Heuristic Knight's Tour attempt:
 1  4 49 20 31  6 33 36 
48 19  2  5 50 35 30  7 
 3 60 47 54 21 32 37 34 
18 53 22 61 46 51  8 29 
23 14 59 52 55 40 45 38 
58 17 56 41 62 43 28  9 
13 24 15 64 11 26 39 44 
16 57 12 25 42 63 10 27 

Moves completed: 64

Heuristic full tours (out of 64 starts): 63
